# Recommender System - Benzer Eyalet Öneri Sistemi

Bir eyalet seçildiğinde kronik hastalık göstergeleri (sigara, obezite, diyabet vb.) bakımından en çok benzeyen eyaletleri öneriyoruz. Sağlık politikası karşılaştırmaları için kullanılabilir.

In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns',100)

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df=pd.read_csv('cdi.csv',low_memory=False)

In [3]:
df.head()

,YearStart,YearEnd,LocationAbbr,LocationDesc,DataSource,Topic,Question,Response,DataValueUnit,DataValueType,DataValue,DataValueAlt,DataValueFootnoteSymbol,DataValueFootnote,LowConfidenceLimit,HighConfidenceLimit,StratificationCategory1,Stratification1,StratificationCategory2,Stratification2,StratificationCategory3,Stratification3,Geolocation,LocationID,TopicID,QuestionID,ResponseID,DataValueTypeID,StratificationCategoryID1,StratificationID1,StratificationCategoryID2,StratificationID2,StratificationCategoryID3,StratificationID3
0,2020,2020,AR,Arkansas,NVSS,Asthma,"Asthma mortality among all people, underlying ...",NaN,Number,Number,20,20,NaN,NaN,NaN,NaN,Sex,Female,NaN,NaN,NaN,NaN,POINT (-92.27449074 34.74865012400045),5,AST,AST01,NaN,NMBR,SEX,SEXF,NaN,NaN,NaN,NaN
1,2019,2019,IA,Iowa,NVSS,Asthma,"Asthma mortality among all people, underlying ...",NaN,Number,Number,32,32,NaN,NaN,NaN,NaN,Sex,Female,NaN,NaN,NaN,NaN,POINT (-93.81649056 42.46940091300047),19,AST,AST01,NaN,NMBR,SEX,SEXF,NaN,NaN,NaN,NaN
2,2021,2021,CO,Colorado,BRFSS,Asthma,Current asthma among adults,NaN,%,Crude Prevalence,8.1,8.1,NaN,NaN,7.2,9,Sex,Male,NaN,NaN,NaN,NaN,POINT (-106.1336109 38.843840757000464),8,AST,AST02,NaN,CRDPREV,SEX,SEXM,NaN,NaN,NaN,NaN
3,2023,2023,KY,Kentucky,BRFSS,Asthma,Current asthma among adults,NaN,%,Crude Prevalence,NaN,NaN,*,No data available,NaN,NaN,Age,Age >=65,NaN,NaN,NaN,NaN,POINT (-84.77497105 37.645970271000465),21,AST,AST02,NaN,CRDPREV,AGE,AGE65P,NaN,NaN,NaN,NaN
4,2023,2023,MD,Maryland,BRFSS,Asthma,Current asthma among adults,NaN,%,Crude Prevalence,10.1,10.1,NaN,NaN,8.8,11.6,Age,Age >=65,NaN,NaN,NaN,NaN,POINT (-76.60926011 39.29058096400047),24,AST,AST02,NaN,CRDPREV,AGE,AGE65P,NaN,NaN,NaN,NaN


In [4]:
df=df[(df['StratificationCategory1']=='Overall')&(df['DataValueType']=='Crude Prevalence')]
df['DataValueAlt']=pd.to_numeric(df['DataValueAlt'],errors='coerce')
df.shape

(14396, 34)

In [5]:
tablo=df.pivot_table(index='LocationDesc',columns='Question',values='DataValueAlt',aggfunc='mean')
tablo=tablo.drop(['United States'],errors='ignore')
tablo.shape

(54, 80)

Çok eksik olan göstergeleri atıp kalan boşlukları o göstergenin ortalamasıyla dolduruyoruz.

In [6]:
tablo=tablo.loc[:,tablo.isnull().mean()<0.3]
tablo=tablo.fillna(tablo.mean())
tablo.shape

(54, 79)

In [7]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
xs=pd.DataFrame(scaler.fit_transform(tablo),index=tablo.index,columns=tablo.columns)

In [8]:
from sklearn.neighbors import NearestNeighbors
nn=NearestNeighbors(metric='cosine',algorithm='brute')
nn.fit(xs.values)

,n_neighbors,5
,radius,1.0
,algorithm,'brute'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


In [9]:
def oner(eyalet,n=5):
    idx=list(xs.index).index(eyalet)
    uzaklik,indeks=nn.kneighbors(xs.iloc[[idx]].values,n_neighbors=n+1)
    return pd.DataFrame({'eyalet':xs.index[indeks[0][1:]],'benzerlik':1-uzaklik[0][1:]})

In [10]:
oner(xs.index[0])

,eyalet,benzerlik
0,Mississippi,0.824389
1,Arkansas,0.812603
2,Louisiana,0.788963
3,Tennessee,0.779845
4,South Carolina,0.755792


### Modeli kaydetme

In [11]:
import pickle
pickle.dump(nn,open('eyalet_nn.pkl','wb'))
pickle.dump(xs,open('eyalet_veri.pkl','wb'))